## LangGraph ReAct Agent with Tools
Learning Objectives:
- Create tools (functions) that LLMs can call
- Implement the ReAct pattern (Reasoning + Acting)
- Build an agent that decides when to use tools

#### 
Real-World Tools:
-----------------
- Database queries
- API calls (weather, stock prices, etc.)
- File operations
- Web searches
- Send emails/notifications
- Execute code
- Image generation
- Data analysis

### Agent Patterns
- Chain of Thoughts (CoT)
- Tree of Thoughts (ToT)
- ReAct

In [1]:
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START,END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage
load_dotenv()

True

In [2]:
llm = ChatOpenAI(temperature=0) #  do experiment with different temperatures


In [ ]:
import my_tools
# examples 
my_tools.calculate.invoke({'expression': '2+2*1.4/23-34'})

[TOOL] calculate ('2+2*1.4/23-34') -> '-31.878260869565217'


-31.878260869565217

In [6]:
eval('2+2*1.4/23-34')

-31.878260869565217

In [23]:
all_tools=[my_tools.calculate,my_tools.get_weather]

from typing_extensions import TypedDict, Annotated
import operator
class AgentState(TypedDict):
    messages: Annotated[list, operator.add]

In [24]:

def agent_node(state: AgentState):

    llm_with_tools = llm.bind_tools(all_tools)

    messages = state['messages']

    response = llm_with_tools.invoke(messages)
    print(response)

    # if hasattr(response, 'tool_calls') and response.tool_calls:
    #     for tc in response.tool_calls:
    #         print(f"[AGENT] called Tool {tc.get('name', '?')} with args {tc.get('args', '?')}")
    # else:
    #     print(f"[AGENT] Responding...")


    return {'messages': [response]}

In [25]:
state = {"messages": [HumanMessage("what is temp in mumbai")]}
result = agent_node(state)

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 321, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DT7yE2Bd2nYtphfKuhYZUkXVB8w3E', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019d77fc-60a0-72a0-b887-2029785e2cc3-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_xQly4RCAsIw4KAQ6ELdWIL43', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 321, 'output_tokens': 15, 'total_tokens': 336, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
